In [ ]:
#@title 1. Cargar los modelos
%pip install -q ultralytics==8.4.192 lap==0.5.13

import sys
from pathlib import Path
from urllib.request import urlretrieve

import torch

version = '80c40095830d21de31cf6b1b70111bd6dc149e0f'
origen = f'https://raw.githubusercontent.com/MauroProto/inspeccion-de-manzanas/{version}'
carpeta = Path('/content') / f'webcam-manzanas-{version[:8]}'
carpeta.mkdir(exist_ok=True)

def descargar(directorio, nombres):
    for nombre in nombres:
        destino = carpeta / nombre
        if not destino.is_file():
            temporal = carpeta / (nombre + '.part')
            urlretrieve(f'{origen}/{directorio}/{nombre}', temporal)
            temporal.replace(destino)

descargar('notebooks/apoyo', ['seguimiento.py', 'inspeccion.py', 'coral.py', 'video.py', 'camara.py', 'camara.js'])
descargar('pesos', ['yolo_manzanas.pt', 'cnn_variedades.pt', 'cnn_danio_visible.pt', 'cnn_confirmacion_danio.pt'])

sys.path.insert(0, str(carpeta))
from inspeccion import Inspeccion
from camara import ver_en_vivo

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
inspeccion = Inspeccion(carpeta / 'yolo_manzanas.pt', carpeta / 'cnn_variedades.pt',
                        carpeta / 'cnn_danio_visible.pt', dispositivo=dispositivo,
                        pesos_confirmacion=carpeta / 'cnn_confirmacion_danio.pt')
equipo = torch.cuda.get_device_name(0) if dispositivo == 'cuda' else 'CPU de Colab'
print(f'Modelos listos en {equipo}. Ejecutá la segunda celda.')

In [ ]:
#@title 2. Encender la webcam
#@markdown Permití el acceso a la cámara. Para elegir una USB, podés escribir parte de su nombre.
nombre_camara = '' #@param {type:"string"}

ver_en_vivo(inspeccion, nombre_camara)